# Coral Reef Bleaching - Gradio Prediction App

Downloads 4 trained models from Google Drive and launches an interactive Gradio app.

**Google Drive:** https://drive.google.com/drive/folders/1pQVpg-ne5EJzaL0BDooCKKvIBquaeIUm

**Important:** All temperature values use **Kelvin** (297–303K range) to match the training dataset.
297K ≈ 24°C | 300K ≈ 27°C | 303K ≈ 30°C | 306K ≈ 33°C

## 0. Install Dependencies

In [1]:
import subprocess, sys
PKGS = ['gradio','scikit-learn','xgboost','pandas','numpy','matplotlib','joblib','gdown']
for pkg in PKGS:
    try: __import__(pkg.replace('-','_'))
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
import gradio as gr
print('Gradio version:', gr.__version__)
print('All packages ready.')

Installing scikit-learn...
Gradio version: 5.50.0
All packages ready.


## 1. Download Models from Google Drive
Downloads all .pkl files from the shared Drive folder into `models/`.

**Folder:** https://drive.google.com/drive/folders/1pQVpg-ne5EJzaL0BDooCKKvIBquaeIUm

Make sure sharing is set to **'Anyone with the link'**.

In [2]:
import gdown, os, shutil, subprocess, sys

GDRIVE_FOLDER_ID = '1pQVpg-ne5EJzaL0BDooCKKvIBquaeIUm'

# NOTE: LR is now a Pipeline model — no separate lr_scaler.pkl
MODEL_FILES = {
    'lr' : ['lr_model.pkl', 'lr_features.pkl', 'lr_metadata.json'],
    'rf' : ['rf_model.pkl', 'rf_features.pkl', 'rf_metadata.json'],
    'svm': ['svm_model.pkl', 'svm_scaler.pkl', 'svm_features.pkl', 'svm_metadata.json'],
    'xgb': ['xgb_model.pkl', 'xgb_features.pkl', 'xgb_metadata.json'],
}

def download_models():
    # Always wipe old files first — prevents Drive file ID caching serving stale models
    if os.path.exists('models'):
        shutil.rmtree('models')
        print('Cleared old models/ folder')
    if os.path.exists('gdrive_models'):
        shutil.rmtree('gdrive_models')
    os.makedirs('models', exist_ok=True)

    print('Downloading models from Google Drive...')
    try:
        gdown.download_folder(
            id=GDRIVE_FOLDER_ID,
            output='gdrive_models',
            quiet=False,
            use_cookies=False,
        )
        for prefix, files in MODEL_FILES.items():
            src_folder = os.path.join('gdrive_models', prefix)
            for fname in files:
                src = os.path.join(src_folder, fname)
                dst = os.path.join('models', fname)
                if os.path.exists(src):
                    shutil.copy2(src, dst)
                    print(f'  Copied: {fname}')
                else:
                    print(f'  WARNING: {fname} not found in Drive')
        if os.path.exists('gdrive_models'):
            shutil.rmtree('gdrive_models')
    except Exception as e:
        print(f'Download failed: {e}')

    print('\nVerification:')
    for prefix, files in MODEL_FILES.items():
        for fname in files:
            path = f'models/{fname}'
            if os.path.exists(path):
                kb = round(os.path.getsize(path)/1024, 1)
                print(f'  [ok] {path:<45} {kb} KB')
            else:
                print(f'  [MISSING] {path}')

download_models()

Retrieving folder contents


Retrieving folder 1lPht_Hu8M15CRg_eYDa0SPy_coPBCfcG lr
Processing file 15bjrTrysLQsuNCy3F8zlP0bU76VEQRCv lr_features.pkl
Processing file 1nD8rM9YTf9b5n0hABhhRg-0DAt34MYmD lr_metadata.json
Processing file 1tA4LmhikaODEKpUvt1qnyOB0sE1f-cf0 lr_model.pkl
Retrieving folder 1NaCAT2yGWZPiZh5kkhRFwD4ubkBTpWR6 rf
Retrieving folder 1_qUF6hYfXeNNE6MyJXR2p3gXgJp0lncv RandomForest
Processing file 1QUgegn2PE1Df0PAQN3PvVctn4OvgA2vZ correlation_heatmap.png
Processing file 1JBEV5tF6tPKBMwfRvzx3-wMDSvpkH8em cv_RF.png
Processing file 1i8WymvZDe12xLnLXnBztV7O_iFkYM0w6 eda_overview.png
Processing file 1lC4i6Y4kSTVU7ryEVZzJ5BMHIFF2yjCL results_Random_Forest.png
Processing file 1dVp5u4It9Am9U78Cc_enFbnvZnwV2Bht rf_feature_importance.png
Processing file 1k83RF5uHndCIMxcD5KY-I2FQ8IDtFTTi rf_features.pkl
Processing file 1v9WCtRJYhXBNWX2coGjt5tPM-igaltSm rf_metadata.json
Processing file 1D27EAG5kQbgWixpVlbWzW3ThMg2-MpZr rf_model.pkl
Processing file 13Ht4lzZrIkDVFi07jpqQ0KEo5Th5eyJR y_prob_Random_Forest.npy
Proce

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=15bjrTrysLQsuNCy3F8zlP0bU76VEQRCv
To: /content/gdrive_models/lr/lr_features.pkl
100%|██████████| 271/271 [00:00<00:00, 162kB/s]
Downloading...
From: https://drive.google.com/uc?id=1nD8rM9YTf9b5n0hABhhRg-0DAt34MYmD
To: /content/gdrive_models/lr/lr_metadata.json
100%|██████████| 753/753 [00:00<00:00, 2.25MB/s]
Downloading...
From: https://drive.google.com/uc?id=1tA4LmhikaODEKpUvt1qnyOB0sE1f-cf0
To: /content/gdrive_models/lr/lr_model.pkl
100%|██████████| 3.55k/3.55k [00:00<00:00, 7.77MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QUgegn2PE1Df0PAQN3PvVctn4OvgA2vZ
To: /content/gdrive_models/rf/RandomForest/correlation_heatmap.png
100%|██████████| 207k/207k [00:00<00:00, 73.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JBEV5tF6tPKBMwfRvzx3-wMDSvpkH8em
To: /content/gdrive_models/rf/RandomForest/cv_RF.png
100%|███

  Copied: lr_model.pkl
  Copied: lr_features.pkl
  Copied: lr_metadata.json
  Copied: rf_model.pkl
  Copied: rf_features.pkl
  Copied: rf_metadata.json
  Copied: svm_model.pkl
  Copied: svm_scaler.pkl
  Copied: svm_features.pkl
  Copied: svm_metadata.json
  Copied: xgb_model.pkl
  Copied: xgb_features.pkl
  Copied: xgb_metadata.json

Verification:
  [ok] models/lr_model.pkl                           3.5 KB
  [ok] models/lr_features.pkl                        0.3 KB
  [ok] models/lr_metadata.json                       0.7 KB
  [ok] models/rf_model.pkl                           70615.4 KB
  [ok] models/rf_features.pkl                        0.3 KB
  [ok] models/rf_metadata.json                       0.5 KB
  [ok] models/svm_model.pkl                          1496.9 KB
  [ok] models/svm_scaler.pkl                         1.5 KB
  [ok] models/svm_features.pkl                       0.3 KB
  [ok] models/svm_metadata.json                      0.5 KB
  [ok] models/xgb_model.pkl                


Download completed


## 2. Load All 4 Models

In [3]:
import joblib, json, os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline as _SkPipeline

MODEL_PREFIXES = {
    'Logistic Regression' : 'lr',
    'Random Forest'       : 'rf',
    'XGBoost'             : 'xgb',
    'SVM (RBF Kernel)'    : 'svm',
}

MODELS = {}; SCALERS = {}; FEAT_DICT = {}; METADATA = {}

for name, prefix in MODEL_PREFIXES.items():
    mpath = f'models/{prefix}_model.pkl'
    fpath = f'models/{prefix}_features.pkl'
    spath = f'models/{prefix}_scaler.pkl'
    jpath = f'models/{prefix}_metadata.json'

    if not os.path.exists(mpath):
        print(f'  SKIPPED {name} -- {mpath} not found')
        continue

    MODELS[name]    = joblib.load(mpath)
    SCALERS[name]   = joblib.load(spath) if os.path.exists(spath) else None
    FEAT_DICT[name] = joblib.load(fpath) if os.path.exists(fpath) else []

    if os.path.exists(jpath):
        with open(jpath) as f:
            METADATA[name] = json.load(f)
        # Metadata features take priority over .pkl features
        if 'features' in METADATA[name]:
            FEAT_DICT[name] = METADATA[name]['features']
    else:
        METADATA[name] = {}

    is_pipe  = isinstance(MODELS[name], _SkPipeline)
    needs_sc = METADATA[name].get('needs_scaling', False)
    size_kb  = round(os.path.getsize(mpath)/1024, 1)
    print(f'  Loaded: {name:<25} '
          f'pipeline={is_pipe} '
          f'needs_scaling={needs_sc} '
          f'features={len(FEAT_DICT[name])} '
          f'size={size_kb}KB')

# Quick LR size check
if os.path.exists('models/lr_model.pkl'):
    lr_kb = round(os.path.getsize('models/lr_model.pkl')/1024, 1)
    if lr_kb < 2:
        print(f'WARNING: lr_model.pkl is {lr_kb}KB -- likely old broken file in Drive')
        print('         Delete old lr/ files from Drive and re-upload new ones')
    else:
        print(f'LR model size {lr_kb}KB -- OK')

print(f'\nModels loaded: {len(MODELS)}/{len(MODEL_PREFIXES)}')

  Loaded: Logistic Regression       pipeline=True needs_scaling=False features=16 size=3.5KB
  Loaded: Random Forest             pipeline=False needs_scaling=False features=16 size=70615.4KB
  Loaded: XGBoost                   pipeline=False needs_scaling=False features=16 size=2927.9KB
  Loaded: SVM (RBF Kernel)          pipeline=False needs_scaling=True features=16 size=1496.9KB
LR model size 3.5KB -- OK

Models loaded: 4/4


/usr/lib/python3.12/pickle.py:1760: UserWarning: [17:47:27] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [17:47:27] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [17:47:27] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


## 3. Prediction Logic & Sample Sites
`predict_with_model()` handles both Pipeline models (LR poly) and standard models (RF, XGBoost, SVM).

**All temperatures are in Kelvin** to match the training dataset.

In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ALL_FEAT_COLS = [
    'ClimSST','Temperature_Mean','Temperature_Minimum','Temperature_Maximum',
    'SSTA','SSTA_DHW','TSA','TSA_DHW','TSA_DHW_Frequency','Windspeed',
    'SSTA_Frequency','SSTA_Frequency_Standard_Deviation','Turbidity_ct',
    'Turbidity','Cyclone_Frequency','Distance','Depth',
    'Latitude_Degrees','Longitude_Degrees','Date_Year',
]

# ── Temperatures are in KELVIN to match training data ──────────────────────
# 297K ≈ 24°C | 299K ≈ 26°C | 300K ≈ 27°C | 302K ≈ 29°C | 303K ≈ 30°C
# TSA_DHW training range: 0.00 to 2.97
SAMPLE_SITES = {
    '-- Select a sample site --':
        [299.5, 300.5, 297.8, 302.0, 0.5, 1.0, 0.8, 1.0, 0.10, 5.0, 0.20, 0.10, 0.05, 0.05, 0.05, 10.0, 5.0, -18.0, 147.0, 2018],
    'Great Barrier Reef 2016 (Bleach)':
        [299.5, 302.0, 298.5, 303.5, 2.1, 2.5, 2.5, 2.5, 0.30, 4.2, 0.40, 0.20, 0.04, 0.04, 0.10, 15.0, 5.0, -18.3, 147.2, 2016],
    'Great Barrier Reef 2020 (Mild)':
        [299.5, 300.5, 297.8, 302.0, 0.8, 1.2, 1.0, 1.2, 0.12, 5.1, 0.20, 0.10, 0.05, 0.05, 0.10, 15.0, 5.0, -18.3, 147.2, 2020],
    'Maldives Healthy (2019)':
        [299.8, 300.2, 298.5, 301.5, 0.3, 0.5, 0.4, 0.5, 0.05, 6.0, 0.10, 0.05, 0.03, 0.03, 0.05,  8.0, 6.0,   4.2,  73.5, 2019],
    'Maldives Bleached (2016)':
        [299.8, 302.5, 298.8, 303.8, 2.8, 2.5, 3.1, 2.5, 0.40, 3.5, 0.50, 0.25, 0.03, 0.03, 0.05,  8.0, 6.0,   4.2,  73.5, 2016],
    'Red Sea Resilient':
        [298.5, 299.0, 297.0, 301.0, 0.1, 0.2, 0.2, 0.2, 0.02, 7.0, 0.05, 0.03, 0.08, 0.08, 0.02, 20.0, 8.0,  22.5,  37.8, 2018],
    'Caribbean Warm Season (2015)':
        [299.3, 301.5, 297.8, 302.8, 1.7, 2.2, 2.0, 2.2, 0.25, 5.5, 0.35, 0.15, 0.06, 0.06, 0.08, 10.0, 4.0,  17.5, -66.0, 2015],
    'Sri Lanka Coast (2016)':
        [299.6, 301.0, 298.0, 302.5, 1.2, 1.8, 1.5, 1.8, 0.18, 4.8, 0.25, 0.12, 0.07, 0.07, 0.06,  5.0, 3.0,   6.9,  81.5, 2016],
    'Deep Protected Reef (Low Risk)':
        [297.5, 298.0, 296.5, 299.5,-0.2, 0.1,-0.1, 0.1, 0.01, 8.0, 0.02, 0.01, 0.12, 0.12, 0.01, 30.0,25.0, -22.0, 114.0, 2017],
    'Turbid Coastal Reef':
        [299.0, 301.2, 297.5, 302.5, 1.9, 2.5, 2.2, 2.5, 0.28, 3.2, 0.38, 0.18, 0.35, 0.35, 0.09,  2.0, 3.0,  -8.5, 115.2, 2016],
    'El Nino Hotspot (2016)':
        [300.5, 302.8, 299.0, 303.8, 2.5, 2.9, 2.8, 2.9, 0.45, 2.8, 0.55, 0.28, 0.02, 0.02, 0.12, 12.0, 4.0, -16.0, 145.5, 2016],
}

COLORS = {
    'Logistic Regression': '#E74C3C',
    'Random Forest'       : '#2ECC71',
    'XGBoost'             : '#F39C12',
    'SVM (RBF Kernel)'    : '#3498DB',
}


def predict_with_model(name, model, X_raw_df):
    """
    Universal predict — works for Pipeline (LR poly) and standard models.
    Pipeline (LR poly): pass raw DataFrame directly — scaler+poly+LR all inside.
    Standard (RF/XGB/SVM): apply scaler if needed, pass numpy array.
    """
    feats    = FEAT_DICT[name]
    scaler   = SCALERS[name]
    needs_sc = METADATA[name].get('needs_scaling', scaler is not None)

    # Align columns to what the model was trained on
    cols = [f for f in feats if f in X_raw_df.columns]
    X    = X_raw_df[cols].copy()

    if isinstance(model, _SkPipeline):
        # Pipeline: scaler + poly + LR all inside — pass raw DataFrame
        pass
    elif needs_sc and scaler is not None:
        n_exp = scaler.n_features_in_
        if X.shape[1] != n_exp:
            # Pad missing features with 0
            X_full = pd.DataFrame(0.0, index=X.index, columns=feats)
            for c in cols:
                X_full[c] = X[c].values
            X = scaler.transform(X_full.values)
        else:
            X = scaler.transform(X.values)
    else:
        X = X.values

    y_pred = np.array(model.predict(X))
    y_prob = np.array(model.predict_proba(X)[:, 1])
    return int(y_pred[0]), float(y_prob[0])


def predict_all_models(*slider_vals):
    if not MODELS:
        return 'No models loaded. Run Cells 1 and 2 first.', None

    inp = dict(zip(ALL_FEAT_COLS, [float(v) for v in slider_vals]))
    X_input = pd.DataFrame([inp])

    names_out = []; probs_out = []; preds_out = []

    for name, model in MODELS.items():
        pred, prob = predict_with_model(name, model, X_input)
        names_out.append(name)
        probs_out.append(prob)
        preds_out.append(pred)

    votes = sum(preds_out)
    avg   = np.mean(probs_out) * 100

    if   votes == 4: cons = 'CRITICAL -- All 4 predict BLEACHING'
    elif votes == 3: cons = 'HIGH RISK -- 3/4 predict bleaching'
    elif votes == 2: cons = 'MODERATE -- 2/4 predict bleaching'
    elif votes == 1: cons = 'LOW RISK -- 1/4 predict bleaching'
    else:            cons = 'SAFE -- All 4 predict NO bleaching'

    txt  = '=' * 52 + '\n'
    txt += '  CORAL REEF BLEACHING PREDICTION\n'
    txt += '=' * 52 + '\n\n'
    for name, prob, pred in zip(names_out, probs_out, preds_out):
        bar  = chr(9608) * int(prob*20) + chr(9617) * (20 - int(prob*20))
        lbl  = 'BLEACHING' if pred == 1 else 'SAFE'
        txt += f'  {name:<25}\n'
        txt += f'  [{bar}] {prob*100:5.1f}%  ->  {lbl}\n\n'
    txt += '-' * 52 + '\n'
    txt += f'  Average probability : {avg:.1f}%\n'
    txt += f'  Votes for bleaching : {votes} / {len(MODELS)}\n'
    txt += f'  CONSENSUS           : {cons}\n'
    txt += '=' * 52

    # Chart
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.patch.set_facecolor('#0f172a')

    ax1 = axes[0]; ax1.set_facecolor('#1e293b')
    bcolors = [COLORS.get(n, '#888') for n in names_out]
    bars = ax1.barh(names_out, [p*100 for p in probs_out],
                    color=bcolors, edgecolor='none', height=0.5)
    ax1.axvline(50, color='white', ls='--', lw=1, alpha=0.6)
    ax1.set_xlim(0, 110)
    ax1.set_xlabel('Bleaching Probability (%)', color='white', fontsize=10)
    ax1.set_title('Prediction by Model', color='white', fontweight='bold', fontsize=11)
    ax1.tick_params(colors='white'); ax1.spines[:].set_visible(False)
    for bar, prob in zip(bars, probs_out):
        ax1.text(min(prob*100+2, 105), bar.get_y()+bar.get_height()/2,
                 f'{prob*100:.1f}%', va='center', color='white',
                 fontsize=10, fontweight='bold')

    ax2 = axes[1]; ax2.set_facecolor('#1e293b'); ax2.set_aspect('equal')
    gc = '#FF4444' if votes >= 3 else '#FFA500' if votes >= 2 else '#2ECC71'
    ax2.pie([avg, 100-avg], colors=[gc, '#2d3748'],
            startangle=90, counterclock=False,
            wedgeprops=dict(width=0.45, edgecolor='#0f172a', linewidth=2))
    ax2.text(0,  0.08, f'{avg:.0f}%', ha='center', va='center',
             color='white', fontsize=24, fontweight='bold')
    ax2.text(0, -0.18, 'Avg Risk', ha='center', color='#94a3b8', fontsize=9)
    ax2.text(0, -0.38, cons.split('--')[0].strip(),
             ha='center', color=gc, fontsize=10, fontweight='bold')
    ax2.set_title('Consensus Gauge', color='white', fontweight='bold', fontsize=11)
    plt.tight_layout(pad=2)
    return txt, fig


def load_sample_site(site_label):
    vals = SAMPLE_SITES.get(site_label,
                            SAMPLE_SITES['-- Select a sample site --'])
    return vals


print(f'Sample sites: {len(SAMPLE_SITES)-1}')
print('Prediction functions ready.')

Sample sites: 10
Prediction functions ready.


## 4. Launch Gradio App
Run this cell. A **public Colab URL** appears below — valid 72 hours.

**Temperature sliders use Kelvin.** Reference: 297K ≈ 24°C | 300K ≈ 27°C | 303K ≈ 30°C | 306K ≈ 33°C

In [5]:
import gradio as gr

# Sliders use Kelvin for temperature features (training data range: ~293–307K)
# DHW values capped at 3.0 to stay within training range (0.00–2.97)
SLIDER_CFG = [
    ('ClimSST - Climatological SST (K)',              293.0, 307.0, 299.5, 0.1),
    ('Temperature_Mean - Mean SST (K)',               293.0, 308.0, 300.5, 0.1),
    ('Temperature_Minimum - Min SST (K)',             290.0, 305.0, 297.8, 0.1),
    ('Temperature_Maximum - Max SST (K)',             295.0, 310.0, 302.0, 0.1),
    ('SSTA - SST Anomaly (K)',                         -3.0,   5.0,   1.0, 0.1),
    ('SSTA_DHW - SST Anomaly DHW',                     0.0,   3.0,   1.0, 0.05),
    ('TSA - Thermal Stress Anomaly (K)',               -2.0,   5.0,   1.2, 0.1),
    ('TSA_DHW - Thermal Stress DHW',                   0.0,   3.0,   1.5, 0.05),
    ('TSA_DHW_Frequency',                              0.0,   1.0,   0.2, 0.01),
    ('Windspeed (m/s)',                                0.0,  15.0,   5.0, 0.1),
    ('SSTA_Frequency',                                 0.0,   1.0,   0.3, 0.01),
    ('SSTA_Frequency_Standard_Deviation',              0.0,   0.5,   0.1, 0.01),
    ('Turbidity_ct - Turbidity',                       0.0,   1.0,  0.05, 0.01),
    ('Turbidity',                                      0.0,   1.0,  0.05, 0.01),
    ('Cyclone_Frequency',                              0.0,   1.0,  0.05, 0.01),
    ('Distance - Distance to Land (km)',               0.0, 100.0,  10.0, 0.5),
    ('Depth - Reef Depth (m)',                         0.0,  50.0,   5.0, 0.5),
    ('Latitude_Degrees',                             -35.0,  35.0, -18.0, 0.1),
    ('Longitude_Degrees',                           -180.0, 180.0, 147.0, 0.1),
    ('Date_Year',                                   1980.0,2024.0, 2016.0, 1.0),
]

SITE_LABELS = list(SAMPLE_SITES.keys())

status_msg = (
    f'Models loaded: {", ".join(MODELS.keys())}'
    if MODELS else 'No models loaded -- run Cells 1 and 2 first'
)

with gr.Blocks(
    theme=gr.themes.Base(primary_hue='green', neutral_hue='slate'),
    title='Coral Reef Bleaching Predictor',
) as demo:

    gr.Markdown(
        '# Coral Reef Bleaching Risk Predictor\n'
        '### LR (Pipeline) | Random Forest | XGBoost | SVM\n'
        '> **Temperature inputs are in Kelvin** — 300K ≈ 27°C | 303K ≈ 30°C'
    )
    gr.Markdown(f'> **{status_msg}**')

    with gr.Row():

        # LEFT: inputs
        with gr.Column(scale=1):

            gr.Markdown('### Sample Reef Sites')
            site_dropdown = gr.Dropdown(
                choices=SITE_LABELS,
                value=SITE_LABELS[0],
                label='Select a reef site to auto-fill all sliders',
            )
            load_btn = gr.Button('Load Site + Predict', variant='primary')

            gr.Markdown('---')
            gr.Markdown('### Manual Ocean Conditions (temperatures in Kelvin)')

            with gr.Accordion('Temperature Features (K)', open=True):
                sl_temp = [gr.Slider(mn, mx, value=v, step=st, label=lbl)
                           for lbl, mn, mx, v, st in SLIDER_CFG[:4]]
            with gr.Accordion('Thermal Stress Features', open=True):
                sl_stress = [gr.Slider(mn, mx, value=v, step=st, label=lbl)
                             for lbl, mn, mx, v, st in SLIDER_CFG[4:9]]
            with gr.Accordion('Wind and Frequency', open=False):
                sl_wind = [gr.Slider(mn, mx, value=v, step=st, label=lbl)
                           for lbl, mn, mx, v, st in SLIDER_CFG[9:12]]
            with gr.Accordion('Water and Location', open=False):
                sl_loc = [gr.Slider(mn, mx, value=v, step=st, label=lbl)
                          for lbl, mn, mx, v, st in SLIDER_CFG[12:]]

            ALL_SLIDERS = sl_temp + sl_stress + sl_wind + sl_loc
            predict_btn = gr.Button('Predict Bleaching Risk',
                                    variant='primary', size='lg')

        # RIGHT: outputs
        with gr.Column(scale=1):
            gr.Markdown('### Prediction Results')
            result_text  = gr.Textbox(
                label='All 4 Model Predictions', lines=18)
            result_chart = gr.Plot(label='Risk Visualization')

    # Sample site reference table
    gr.Markdown('---')
    gr.Markdown('### Sample Site Reference (temperatures in Kelvin)')
    table_rows = [
        '| # | Site | ClimSST (K) | Temp_Mean (K) | SSTA | TSA_DHW | Risk | Year |',
        '|---|---|---|---|---|---|---|---|',
    ]
    risk_map = {
        'Great Barrier Reef 2016 (Bleach)'   : 'CRITICAL',
        'Great Barrier Reef 2020 (Mild)'     : 'MODERATE',
        'Maldives Healthy (2019)'            : 'SAFE',
        'Maldives Bleached (2016)'           : 'CRITICAL',
        'Red Sea Resilient'                  : 'SAFE',
        'Caribbean Warm Season (2015)'       : 'HIGH',
        'Sri Lanka Coast (2016)'             : 'HIGH',
        'Deep Protected Reef (Low Risk)'     : 'SAFE',
        'Turbid Coastal Reef'                : 'MODERATE',
        'El Nino Hotspot (2016)'             : 'CRITICAL',
    }
    for i, (site, vals) in enumerate(
            [(s, v) for s, v in SAMPLE_SITES.items()
             if s != '-- Select a sample site --'], 1):
        risk = risk_map.get(site, '--')
        table_rows.append(
            f'| {i} | {site} | {vals[0]} | {vals[1]}'
            f' | {vals[4]} | {vals[7]} | {risk} | {int(vals[19])} |'
        )
    gr.Markdown('\n'.join(table_rows))

    gr.Markdown(
        '---\n'
        '**How to use:**\n'
        '1. Pick a site from dropdown → sliders auto-fill\n'
        '2. Click **Load Site + Predict** to run all 4 models\n'
        '3. Or adjust sliders manually → click **Predict Bleaching Risk**\n\n'
        '**Key risk indicators:** TSA_DHW > 2.5 = high bleaching risk | '
        'Temp_Mean > 302K (~29°C) + SSTA > 1.5 = elevated risk\n\n'
        '**Temperature reference:** 297K=24°C | 300K=27°C | 303K=30°C | 306K=33°C'
    )

    # Events
    site_dropdown.change(
        fn=load_sample_site,
        inputs=[site_dropdown],
        outputs=ALL_SLIDERS,
    )

    def load_and_predict(site_label):
        vals = load_sample_site(site_label)
        return predict_all_models(*vals)

    load_btn.click(
        fn=load_and_predict,
        inputs=[site_dropdown],
        outputs=[result_text, result_chart],
    )

    predict_btn.click(
        fn=predict_all_models,
        inputs=ALL_SLIDERS,
        outputs=[result_text, result_chart],
    )

print('Launching Gradio app...')
print('Public URL will appear below (valid 72 hours on Colab)')
demo.launch(share=True, debug=False, show_error=True, server_name='0.0.0.0')

/tmp/ipykernel_6939/1245884493.py:35: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Launching Gradio app...
Public URL will appear below (valid 72 hours on Colab)
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6824a0b05a1a1907e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
